In [1]:
import sys

sys.path.append("../")

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
from src.model.models import CheckersNetV2

model_name = "checkers3.2"
model_checkpoint_path = f"../models/{model_name}.pt"

checkers_model = CheckersNetV2(
    input_planes=18,
    policy_planes=20480,
    channels=192,
    num_blocks=24,
    temperature=1,
    use_attention=True,
)

model_state_dict = torch.load(f=model_checkpoint_path, map_location=device)

checkers_model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [4]:
from torchinfo import summary

summary(
    model=checkers_model,
    input_size=(64, 18, 8, 8),  # make sure this is "input_size", not "input_shape"
    # col_names=["input_size"], # uncomment for smaller output
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"],
)

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
CheckersNetV2 (CheckersNetV2)            [64, 18, 8, 8]       [64, 20480]          --                   True
├─Conv2d (conv_input)                    [64, 18, 8, 8]       [64, 192, 8, 8]      31,296               True
├─BatchNorm2d (bn_input)                 [64, 192, 8, 8]      [64, 192, 8, 8]      384                  True
├─ModuleList (res_blocks)                --                   --                   --                   True
│    └─BottleneckBlock (0)               [64, 192, 8, 8]      [64, 192, 8, 8]      --                   True
│    │    └─Conv2d (conv1)               [64, 192, 8, 8]      [64, 48, 8, 8]       9,264                True
│    │    └─BatchNorm2d (bn1)            [64, 48, 8, 8]       [64, 48, 8, 8]       96                   True
│    │    └─Conv2d (conv2)               [64, 48, 8, 8]       [64, 48, 8, 8]       20,784               True
│    │    └─Ba

## Test the Chess Engine


In [5]:
import torch

from src.common import tools
from src.evaluate.mcts import MCTS

# ── MCTS configuration ─────────────────────────────────────────────────────────
NUM_SIMULATIONS = 800  # increase for stronger play, decrease for faster moves
C_PUCT = 2.5  # exploration constant

mcts = MCTS(
    model=checkers_model,
    num_simulations=NUM_SIMULATIONS,
    c_puct=C_PUCT,
    device=str(device),
)


def get_model_move(board, model, device):
    """
    Get the best legal move using MCTS guided by the neural network.

    MCTS runs NUM_SIMULATIONS playouts from the current position. Each playout
    uses the policy head for move priors and the value head for leaf evaluation.
    The move with the highest visit count is returned.

    Args:
        board:  chess.Board object representing the current position.
        model:  trained chess model (used internally by the MCTS instance).
        device: torch device (cpu/cuda).

    Returns:
        tuple: (chess.Move or None, visit_share float, value float)
    """
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None, 0.0, 0.0

    # Switch to eval mode before any model calls so BatchNorm uses its
    # accumulated running statistics rather than single-sample batch stats.
    model.eval()

    # Run MCTS — returns {move: visit_count}
    visit_counts = mcts.run(board)

    # Pick the move with the most visits
    best_move = max(visit_counts, key=visit_counts.get)
    total_visits = sum(visit_counts.values())
    visit_share = visit_counts[best_move] / total_visits if total_visits > 0 else 0.0

    # Quick network eval for the position value displayed in the info panel
    position_tensor = tools.encode_fen_pos(board.fen()).unsqueeze(0).to(device)
    with torch.no_grad():
        _, value_tensor = model(position_tensor)
    value = value_tensor.item()

    eval_cp = value * 1000
    print(f"Position evaluation: {eval_cp:.1f} cp")
    print(
        f"Selected move: {best_move}  "
        f"(visits: {visit_counts[best_move]}/{total_visits},  "
        f"visit share: {visit_share:.4f})"
    )

    return best_move, visit_share, value

## Interactive Chess Game


In [6]:
import os

import chess
import pygame as p

# ── CONSTANTS ──────────────────────────────────────────────────────────────────
BOARD_WIDTH = BOARD_HEIGHT = 512
INFO_PANEL_WIDTH = 280
DIMENSION = 8
SQ_SIZE = BOARD_HEIGHT // DIMENSION
MAX_FPS = 30

# Colours
LIGHT_SQ = (237, 238, 209)
DARK_SQ = (119, 153, 82)
HIGHLIGHT_COLOR = (186, 202, 68)
LAST_MOVE_COLOR = (246, 246, 105, 120)
CHECK_COLOR = (235, 97, 80)
INFO_BG = (48, 46, 43)
INFO_TEXT = (200, 200, 200)
INFO_ACCENT = (129, 182, 76)

PIECE_IMAGES = {}
PIECE_MAP = {
    chess.PAWN: "p",
    chess.ROOK: "R",
    chess.KNIGHT: "N",
    chess.BISHOP: "B",
    chess.QUEEN: "Q",
    chess.KING: "K",
}

# Path to piece images
PIECE_IMG_DIR = os.path.join(os.path.dirname(os.getcwd()), "chess_pieces")


def load_images():
    """Load piece images from chess_pieces/ folder."""
    for color_prefix, color_name in [("w", "white"), ("b", "black")]:
        for piece_char in ["p", "R", "N", "B", "Q", "K"]:
            key = color_prefix + piece_char
            path = os.path.join(PIECE_IMG_DIR, f"{key}.png")
            img = p.image.load(path)
            PIECE_IMAGES[key] = p.transform.smoothscale(img, (SQ_SIZE, SQ_SIZE))


def piece_to_img_key(piece: chess.Piece) -> str:
    """Convert python-chess Piece to image key like 'wK', 'bp'."""
    color_prefix = "w" if piece.color == chess.WHITE else "b"
    return color_prefix + PIECE_MAP[piece.piece_type]


def sq_to_screen(sq, flipped):
    """Convert a chess square to (screen_row, screen_col).

    Standard (flipped=False): White at bottom — rank 0 → row 7, file 0 → col 0.
    Flipped  (flipped=True):  Black at bottom — rank 7 → row 7, file 7 → col 0.
    """
    rank = chess.square_rank(sq)
    file = chess.square_file(sq)
    if flipped:
        return rank, 7 - file
    return 7 - rank, file


def screen_to_sq(col, row, flipped):
    """Convert (screen_col, screen_row) back to a chess square."""
    if flipped:
        return chess.square(7 - col, row)
    return chess.square(col, 7 - row)


def draw_board(screen):
    """Draw the chequered board."""
    for r in range(DIMENSION):
        for c in range(DIMENSION):
            color = LIGHT_SQ if (r + c) % 2 == 0 else DARK_SQ
            p.draw.rect(
                screen, color, p.Rect(c * SQ_SIZE, r * SQ_SIZE, SQ_SIZE, SQ_SIZE)
            )


def draw_pieces(screen, board: chess.Board, flipped: bool):
    """Draw pieces on the board from python-chess board state."""
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece:
            row, col = sq_to_screen(sq, flipped)
            key = piece_to_img_key(piece)
            screen.blit(
                PIECE_IMAGES[key],
                p.Rect(col * SQ_SIZE, row * SQ_SIZE, SQ_SIZE, SQ_SIZE),
            )


def highlight_squares(
    screen,
    board: chess.Board,
    selected_sq,
    valid_move_squares,
    last_move,
    flipped: bool,
):
    """Highlight selected square, legal destinations, and last move."""
    # Highlight last move
    if last_move:
        for sq in [last_move.from_square, last_move.to_square]:
            row, col = sq_to_screen(sq, flipped)
            s = p.Surface((SQ_SIZE, SQ_SIZE), p.SRCALPHA)
            s.fill(LAST_MOVE_COLOR)
            screen.blit(s, (col * SQ_SIZE, row * SQ_SIZE))

    # Highlight king in check
    if board.is_check():
        king_sq = board.king(board.turn)
        if king_sq is not None:
            row, col = sq_to_screen(king_sq, flipped)
            s = p.Surface((SQ_SIZE, SQ_SIZE), p.SRCALPHA)
            s.fill((*CHECK_COLOR, 160))
            screen.blit(s, (col * SQ_SIZE, row * SQ_SIZE))

    # Highlight selected square
    if selected_sq is not None:
        row, col = sq_to_screen(selected_sq, flipped)
        s = p.Surface((SQ_SIZE, SQ_SIZE), p.SRCALPHA)
        s.fill((*HIGHLIGHT_COLOR, 180))
        screen.blit(s, (col * SQ_SIZE, row * SQ_SIZE))

    # Show legal move dots
    for sq in valid_move_squares:
        row, col = sq_to_screen(sq, flipped)
        center = (col * SQ_SIZE + SQ_SIZE // 2, row * SQ_SIZE + SQ_SIZE // 2)
        target_piece = board.piece_at(sq)
        if target_piece:
            # Draw ring for captures
            p.draw.circle(screen, (0, 0, 0, 80), center, SQ_SIZE // 2, 4)
        else:
            # Draw dot for quiet moves
            s = p.Surface((SQ_SIZE, SQ_SIZE), p.SRCALPHA)
            p.draw.circle(s, (0, 0, 0, 60), (SQ_SIZE // 2, SQ_SIZE // 2), SQ_SIZE // 6)
            screen.blit(s, (col * SQ_SIZE, row * SQ_SIZE))


def draw_info_panel(
    screen,
    board: chess.Board,
    ai_visit_share,
    ai_value,
    ai_move_str,
    move_log,
    status_msg,
    human_color,
):
    """Draw the info panel on the right showing AI stats and move history."""
    panel_rect = p.Rect(BOARD_WIDTH, 0, INFO_PANEL_WIDTH, BOARD_HEIGHT)
    p.draw.rect(screen, INFO_BG, panel_rect)

    # ── Title ──
    title_font = p.font.SysFont("Segoe UI", 22, bold=True)
    title_surf = title_font.render(model_name.capitalize(), True, p.Color("white"))
    screen.blit(title_surf, (BOARD_WIDTH + 15, 12))

    # Divider
    p.draw.line(
        screen,
        (80, 80, 80),
        (BOARD_WIDTH + 10, 42),
        (BOARD_WIDTH + INFO_PANEL_WIDTH - 10, 42),
    )

    info_font = p.font.SysFont("Segoe UI", 15)
    small_font = p.font.SysFont("Consolas", 13)
    y = 52

    # ── AI Info Section ──
    section_font = p.font.SysFont("Segoe UI", 16, bold=True)
    screen.blit(
        section_font.render("AI Analysis", True, INFO_ACCENT), (BOARD_WIDTH + 15, y)
    )
    y += 28

    # AI move
    label_color = (160, 160, 160)
    screen.blit(
        info_font.render("Last AI move:", True, label_color), (BOARD_WIDTH + 15, y)
    )
    move_text = ai_move_str if ai_move_str else "—"
    screen.blit(
        info_font.render(move_text, True, p.Color("white")), (BOARD_WIDTH + 130, y)
    )
    y += 24

    # MCTS visit share
    screen.blit(
        info_font.render("Visit share:", True, label_color), (BOARD_WIDTH + 15, y)
    )
    share_text = f"{ai_visit_share:.4f}" if ai_visit_share is not None else "—"
    screen.blit(
        info_font.render(share_text, True, p.Color("white")), (BOARD_WIDTH + 130, y)
    )
    y += 24

    # Position value
    screen.blit(
        info_font.render("Position val:", True, label_color), (BOARD_WIDTH + 15, y)
    )
    if ai_value is not None:
        val_color = (
            INFO_ACCENT
            if ai_value > 0.05
            else (CHECK_COLOR if ai_value < -0.05 else INFO_TEXT)
        )
        val_text = f"{ai_value:+.3f}"
    else:
        val_color = INFO_TEXT
        val_text = "—"
    screen.blit(info_font.render(val_text, True, val_color), (BOARD_WIDTH + 130, y))
    y += 24

    # Eval bar (centipawns)
    screen.blit(
        info_font.render("Eval (cp):", True, label_color), (BOARD_WIDTH + 15, y)
    )
    if ai_value is not None:
        cp = ai_value * 1000
        cp_text = f"{cp:+.0f}"
    else:
        cp_text = "—"
    screen.blit(
        info_font.render(cp_text, True, p.Color("white")), (BOARD_WIDTH + 130, y)
    )
    y += 30

    # Visual eval bar
    bar_x = BOARD_WIDTH + 15
    bar_w = INFO_PANEL_WIDTH - 30
    bar_h = 14
    p.draw.rect(screen, (60, 60, 60), (bar_x, y, bar_w, bar_h), border_radius=3)
    if ai_value is not None:
        fill_ratio = max(0.0, min(1.0, (ai_value + 1) / 2))  # map [-1,1] to [0,1]
        fill_w = int(bar_w * fill_ratio)
        bar_color = INFO_ACCENT if fill_ratio >= 0.5 else CHECK_COLOR
        p.draw.rect(screen, bar_color, (bar_x, y, fill_w, bar_h), border_radius=3)
    y += 28

    # Divider
    p.draw.line(
        screen,
        (80, 80, 80),
        (BOARD_WIDTH + 10, y),
        (BOARD_WIDTH + INFO_PANEL_WIDTH - 10, y),
    )
    y += 10

    # ── Game Status ──
    screen.blit(
        section_font.render("Game Status", True, INFO_ACCENT), (BOARD_WIDTH + 15, y)
    )
    y += 24
    turn_text = "White to move" if board.turn == chess.WHITE else "Black to move"
    screen.blit(
        info_font.render(turn_text, True, p.Color("white")), (BOARD_WIDTH + 15, y)
    )
    y += 22
    screen.blit(
        info_font.render(f"Move: {board.fullmove_number}", True, label_color),
        (BOARD_WIDTH + 15, y),
    )
    y += 22
    # Show which color the human is playing
    human_color_text = "You: White" if human_color == chess.WHITE else "You: Black"
    screen.blit(
        info_font.render(human_color_text, True, INFO_ACCENT),
        (BOARD_WIDTH + 15, y),
    )
    y += 28

    if status_msg:
        status_font = p.font.SysFont("Segoe UI", 16, bold=True)
        screen.blit(
            status_font.render(status_msg, True, p.Color("yellow")),
            (BOARD_WIDTH + 15, y),
        )
        y += 26

    # Divider
    p.draw.line(
        screen,
        (80, 80, 80),
        (BOARD_WIDTH + 10, y),
        (BOARD_WIDTH + INFO_PANEL_WIDTH - 10, y),
    )
    y += 10

    # ── Move Log ──
    screen.blit(
        section_font.render("Move Log", True, INFO_ACCENT), (BOARD_WIDTH + 15, y)
    )
    y += 24

    # Show moves in pairs
    moves = list(board.move_stack)
    move_texts = []
    replay = chess.Board()
    for m in moves:
        san = replay.san(m)
        move_texts.append(san)
        replay.push(m)

    max_display_lines = (BOARD_HEIGHT - y - 52) // 18
    lines = []
    for i in range(0, len(move_texts), 2):
        num = i // 2 + 1
        white_move = move_texts[i]
        black_move = move_texts[i + 1] if i + 1 < len(move_texts) else ""
        lines.append(f"{num}. {white_move}  {black_move}")

    # Scroll to show latest moves
    if len(lines) > max_display_lines:
        lines = lines[-max_display_lines:]

    for line in lines:
        screen.blit(small_font.render(line, True, INFO_TEXT), (BOARD_WIDTH + 15, y))
        y += 18

    # ── Controls ──
    controls_y = BOARD_HEIGHT - 48
    ctrl_font = p.font.SysFont("Segoe UI", 12)
    screen.blit(
        ctrl_font.render("Z = Undo  |  R = Reset  |  Q = Quit", True, (120, 120, 120)),
        (BOARD_WIDTH + 15, controls_y),
    )
    screen.blit(
        ctrl_font.render("S = Switch color (resets game)", True, (120, 120, 120)),
        (BOARD_WIDTH + 15, controls_y + 16),
    )


def get_promotion_choice(screen, human_color):
    """Show a promotion popup and return the chosen piece type."""
    overlay = p.Surface((BOARD_WIDTH, BOARD_HEIGHT), p.SRCALPHA)
    overlay.fill((0, 0, 0, 150))
    screen.blit(overlay, (0, 0))

    font = p.font.SysFont("Segoe UI", 24, bold=True)
    text = font.render("Promote to:", True, p.Color("white"))
    screen.blit(text, (BOARD_WIDTH // 2 - text.get_width() // 2, 180))

    color_prefix = "w" if human_color == chess.WHITE else "b"
    pieces = [
        (f"{color_prefix}Q", chess.QUEEN),
        (f"{color_prefix}R", chess.ROOK),
        (f"{color_prefix}B", chess.BISHOP),
        (f"{color_prefix}N", chess.KNIGHT),
    ]
    buttons = []
    bx = (BOARD_WIDTH - 4 * 80) // 2
    for i, (img_key, pt) in enumerate(pieces):
        rect = p.Rect(bx + i * 90, 230, 80, 80)
        p.draw.rect(screen, LIGHT_SQ, rect, border_radius=6)
        img = p.transform.smoothscale(PIECE_IMAGES[img_key], (64, 64))
        screen.blit(img, (rect.x + 8, rect.y + 8))
        buttons.append((rect, pt))

    p.display.flip()

    while True:
        for e in p.event.get():
            if e.type == p.QUIT:
                return chess.QUEEN
            if e.type == p.MOUSEBUTTONDOWN:
                for rect, pt in buttons:
                    if rect.collidepoint(e.pos):
                        return pt


def draw_end_game_text(screen, text):
    """Draw an end-game message overlay."""
    overlay = p.Surface((BOARD_WIDTH, BOARD_HEIGHT), p.SRCALPHA)
    overlay.fill((0, 0, 0, 160))
    screen.blit(overlay, (0, 0))

    font = p.font.SysFont("Segoe UI", 36, bold=True)
    text_surf = font.render(text, True, p.Color("white"))
    x = BOARD_WIDTH // 2 - text_surf.get_width() // 2
    y = BOARD_HEIGHT // 2 - text_surf.get_height() // 2
    screen.blit(text_surf, (x, y))

    sub_font = p.font.SysFont("Segoe UI", 18)
    sub_text = sub_font.render("Press R to reset or Q to quit", True, (200, 200, 200))
    screen.blit(sub_text, (BOARD_WIDTH // 2 - sub_text.get_width() // 2, y + 50))


def main():
    p.init()
    screen = p.display.set_mode((BOARD_WIDTH + INFO_PANEL_WIDTH, BOARD_HEIGHT))
    p.display.set_caption("Checkers AI - Chess")
    clock = p.time.Clock()
    load_images()

    board = chess.Board()
    selected_sq = None
    valid_move_squares = []
    last_move = None
    game_over = False
    status_msg = ""

    # AI tracking
    ai_visit_share = None
    ai_value = None
    ai_move_str = None
    ai_thinking = False

    # Human plays White by default, AI plays Black
    HUMAN_COLOR = chess.WHITE
    AI_COLOR = chess.BLACK

    running = True
    while running:
        flipped = HUMAN_COLOR == chess.BLACK
        human_turn = board.turn == HUMAN_COLOR and not game_over
        ai_turn = board.turn == AI_COLOR and not game_over

        for e in p.event.get():
            if e.type == p.QUIT:
                running = False
                break

            if e.type == p.KEYDOWN:
                # Undo: press Z (undoes both AI + human move)
                if e.key == p.K_z and not ai_thinking:
                    if len(board.move_stack) >= 2:
                        board.pop()  # undo AI move
                        board.pop()  # undo human move
                        last_move = board.move_stack[-1] if board.move_stack else None
                        selected_sq = None
                        valid_move_squares = []
                        game_over = False
                        status_msg = ""
                    elif len(board.move_stack) == 1:
                        board.pop()
                        last_move = None
                        selected_sq = None
                        valid_move_squares = []
                        game_over = False
                        status_msg = ""

                # Reset: press R
                if e.key == p.K_r and not ai_thinking:
                    board.reset()
                    selected_sq = None
                    valid_move_squares = []
                    last_move = None
                    game_over = False
                    status_msg = ""
                    ai_visit_share = None
                    ai_value = None
                    ai_move_str = None

                # Switch color: press S (resets game and swaps colors)
                if e.key == p.K_s and not ai_thinking:
                    HUMAN_COLOR = (
                        chess.BLACK if HUMAN_COLOR == chess.WHITE else chess.WHITE
                    )
                    AI_COLOR = chess.BLACK if AI_COLOR == chess.WHITE else chess.WHITE
                    board.reset()
                    selected_sq = None
                    valid_move_squares = []
                    last_move = None
                    game_over = False
                    status_msg = ""
                    ai_visit_share = None
                    ai_value = None
                    ai_move_str = None

                # Quit: press Q
                if e.key == p.K_q:
                    running = False
                    break

            # Handle mouse clicks for human moves
            if e.type == p.MOUSEBUTTONDOWN and human_turn and not ai_thinking:
                mx, my = e.pos
                if mx < BOARD_WIDTH:  # Click is on the board
                    col = mx // SQ_SIZE
                    row = my // SQ_SIZE
                    clicked_sq = screen_to_sq(col, row, flipped)

                    if selected_sq is None:
                        # Select a piece
                        piece = board.piece_at(clicked_sq)
                        if piece and piece.color == HUMAN_COLOR:
                            selected_sq = clicked_sq
                            # Find legal moves from this square
                            valid_move_squares = []
                            for move in board.legal_moves:
                                if move.from_square == selected_sq:
                                    valid_move_squares.append(move.to_square)
                    else:
                        # Try to make a move
                        if clicked_sq == selected_sq:
                            # Deselect
                            selected_sq = None
                            valid_move_squares = []
                        elif clicked_sq in valid_move_squares:
                            # Check for promotion
                            piece = board.piece_at(selected_sq)
                            promotion = None
                            if piece and piece.piece_type == chess.PAWN:
                                target_rank = chess.square_rank(clicked_sq)
                                if (
                                    piece.color == chess.WHITE and target_rank == 7
                                ) or (piece.color == chess.BLACK and target_rank == 0):
                                    promotion = get_promotion_choice(
                                        screen, HUMAN_COLOR
                                    )

                            move = chess.Move(
                                selected_sq, clicked_sq, promotion=promotion
                            )
                            if move in board.legal_moves:
                                board.push(move)
                                last_move = move
                                selected_sq = None
                                valid_move_squares = []

                                # Check game end
                                if board.is_checkmate():
                                    status_msg = "Checkmate! You win!"
                                    game_over = True
                                elif board.is_stalemate():
                                    status_msg = "Stalemate! Draw."
                                    game_over = True
                                elif board.is_insufficient_material():
                                    status_msg = "Draw - insufficient material."
                                    game_over = True
                                elif board.can_claim_threefold_repetition():
                                    status_msg = "Draw - threefold repetition."
                                    game_over = True
                        else:
                            # Clicked on another own piece -> reselect
                            piece = board.piece_at(clicked_sq)
                            if piece and piece.color == HUMAN_COLOR:
                                selected_sq = clicked_sq
                                valid_move_squares = []
                                for move in board.legal_moves:
                                    if move.from_square == selected_sq:
                                        valid_move_squares.append(move.to_square)
                            else:
                                selected_sq = None
                                valid_move_squares = []

        # ── AI Move ──
        if ai_turn and not ai_thinking and not game_over:
            ai_thinking = True
            status_msg = f"AI thinking... ({NUM_SIMULATIONS} sims)"

            # Draw the "thinking" state before computing
            draw_board(screen)
            highlight_squares(
                screen, board, selected_sq, valid_move_squares, last_move, flipped
            )
            draw_pieces(screen, board, flipped)
            draw_info_panel(
                screen,
                board,
                ai_visit_share,
                ai_value,
                ai_move_str,
                [],
                status_msg,
                HUMAN_COLOR,
            )
            p.display.flip()

            # Get AI move using MCTS
            best_move, visit_share, value = get_model_move(
                board, checkers_model, device
            )

            if best_move and best_move in board.legal_moves:
                ai_move_str = board.san(best_move)
                board.push(best_move)
                last_move = best_move
                ai_visit_share = visit_share
                ai_value = value
            else:
                # Fallback: pick a random legal move if MCTS fails
                import random

                legal = list(board.legal_moves)
                if legal:
                    fallback = random.choice(legal)
                    ai_move_str = board.san(fallback)
                    board.push(fallback)
                    last_move = fallback
                    ai_visit_share = 0.0
                    ai_value = value if value else 0.0

            ai_thinking = False
            status_msg = ""

            # Check game end after AI move
            if board.is_checkmate():
                status_msg = "Checkmate! AI wins!"
                game_over = True
            elif board.is_stalemate():
                status_msg = "Stalemate! Draw."
                game_over = True
            elif board.is_insufficient_material():
                status_msg = "Draw - insufficient material."
                game_over = True
            elif board.can_claim_threefold_repetition():
                status_msg = "Draw - threefold repetition."
                game_over = True

        # ── Drawing ──
        draw_board(screen)
        highlight_squares(
            screen, board, selected_sq, valid_move_squares, last_move, flipped
        )
        draw_pieces(screen, board, flipped)
        draw_info_panel(
            screen,
            board,
            ai_visit_share,
            ai_value,
            ai_move_str,
            [],
            status_msg,
            HUMAN_COLOR,
        )

        if game_over and status_msg:
            draw_end_game_text(screen, status_msg)

        p.display.flip()
        clock.tick(MAX_FPS)

    p.quit()


# Run the game
main()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Position evaluation: -22.6 cp
Selected move: e7e5  (visits: 270/800,  visit share: 0.3375)
